# Train Siamese UNet on LEVIR-CD (Google Colab)
Runtime → Change runtime type → **GPU**. Upload `GeoAI-Satellite-Change-Detection-main.zip` and `LEVIR-CD.zip` (A/B/label folders for train/val/test) to your Google Drive first.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!unzip -q -o /content/drive/MyDrive/GeoAI-Satellite-Change-Detection-main.zip -d /content
%cd /content/GeoAI-Satellite-Change-Detection-main
!mkdir -p data/raw && unzip -q -o /content/drive/MyDrive/LEVIR-CD.zip -d data/raw
!ls data/raw/LEVIR-CD/*

Sanity check: overfit 8 patches. Val F1 should climb to ~1.0 within a few epochs.

In [ ]:
!python train.py --overfit 8 --epochs 15 --batch-size 4 --workers 2 --out models/overfit.pth

Full training. Checkpoints go to Drive so a Colab disconnect doesn't lose them; re-run with `--resume` to continue.

In [ ]:
!mkdir -p /content/drive/MyDrive/geoai_models
!python train.py --epochs 100 --batch-size 16 --amp --workers 2 --out /content/drive/MyDrive/geoai_models/siamese_unet.pth
# resume after disconnect:
# !python train.py --epochs 100 --batch-size 16 --amp --workers 2 --out /content/drive/MyDrive/geoai_models/siamese_unet.pth --resume /content/drive/MyDrive/geoai_models/last.pt

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
log = pd.read_csv('runs/train_log.csv')
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
log.plot(x='epoch', y=['train_loss', 'val_loss'], ax=ax[0]); ax[0].set_title('Loss')
log.plot(x='epoch', y=['precision', 'recall', 'f1', 'iou'], ax=ax[1]); ax[1].set_title('Validation (change class)')
plt.show()

In [ ]:
!python evaluate.py --weights /content/drive/MyDrive/geoai_models/siamese_unet.pth --save-vis 12 --workers 2

In [ ]:
from IPython.display import Image, display
import glob
for p in sorted(glob.glob('runs/vis/*.png'))[:6]:
    display(Image(p))  # A | B | GT | pred (white TP, red FP, green FN)